# Mini-GPT Architecture Walkthrough

This notebook explains the pieces of the from-scratch GPT-style model implemented in `src/minigpt/`.

## Tokenization

Text is converted to token ids by a BPE tokenizer trained with the `tokenizers` library. The Transformer itself only sees integer token ids shaped `[batch_size, sequence_length]`.

## Causal Masking

Decoder-only language models predict the next token without seeing future tokens. The attention matrix has shape `[batch, heads, sequence, sequence]`, and a lower-triangular mask sets future positions to `-inf` before softmax.

In [ ]:
import torch

block_size = 8
mask = torch.tril(torch.ones(block_size, block_size, dtype=torch.bool))
mask

## Self-Attention Shapes

The model projects hidden states into Q, K, and V, then reshapes from `[B, T, C]` to `[B, n_head, T, head_dim]`. Attention scores are `Q @ K.T / sqrt(head_dim)`, then probabilities multiply V.

## Transformer Block

Each block uses pre-LayerNorm, causal self-attention, an MLP with GELU, dropout, and residual connections.

In [ ]:
from minigpt.config import ModelConfig
from minigpt.model import GPT

config = ModelConfig(vocab_size=256, block_size=16, n_layer=2, n_head=2, n_embd=64)
model = GPT(config)
input_ids = torch.randint(0, config.vocab_size, (2, config.block_size))
targets = torch.randint(0, config.vocab_size, (2, config.block_size))
logits, loss = model(input_ids, targets)
logits.shape, loss

## Logits, Loss, and Generation

The final head returns logits shaped `[batch_size, sequence_length, vocab_size]`. Cross-entropy compares those logits to shifted next-token targets. Generation repeatedly crops to the context window, predicts the last-position logits, samples a token, and appends it.